In [1]:
!pip install -q gensim
import glob, time, joblib
from pathlib import Path
import numpy as np
import polars as pl

FEAT = sorted({p.rsplit("/", 1)[0] for p in glob.glob("/kaggle/input/**/train_text.parquet", recursive=True)})[0]
cols = ["complaint_id", "product", "text_tfidf"]
train = pl.read_parquet(f"{FEAT}/train_text.parquet", columns=cols)
val   = pl.read_parquet(f"{FEAT}/val_text.parquet", columns=cols)
print(train.shape, val.shape)

CACHE = Path("/kaggle/working/cache"); CACHE.mkdir(parents=True, exist_ok=True)
REPORTS = Path("/kaggle/working/reports_l3"); REPORTS.mkdir(parents=True, exist_ok=True)

def cached(name, fn):
    """Recharge 'name' depuis working/cache ou n'importe quel input, sinon calcule."""
    candidates = [CACHE / name] + [Path(p) for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)]
    for path in candidates:
        if path.exists():
            obj = joblib.load(path)
            if path.parent != CACHE:
                joblib.dump(obj, CACHE / name)
            print(f"[cache] {name} rechargé")
            return obj
    t0 = time.time()
    obj = fn()
    joblib.dump(obj, CACHE / name)
    print(f"[cache] {name} calculé et sauvegardé ({time.time() - t0:.0f} s)")
    return obj

(100000, 3) (50000, 3)


In [2]:
sentences = [t.split() for t in train["text_tfidf"].to_list()]

n_tokens = sum(len(s) for s in sentences)
vocab = {}
for s in sentences:
    for w in s:
        vocab[w] = vocab.get(w, 0) + 1

print(f"Plaintes       : {len(sentences):,}")
print(f"Mots au total  : {n_tokens:,}")
print(f"Mots différents: {len(vocab):,}")
print(f"Mots vus >= 5 fois : {sum(1 for c in vocab.values() if c >= 5):,}")

print("\nExemple de plainte transformée :")
print(sentences[0][:30])

print("\nLes 20 mots les plus fréquents :")
print(sorted(vocab.items(), key=lambda x: -x[1])[:20])

Plaintes       : 100,000
Mots au total  : 9,785,590
Mots différents: 58,780
Mots vus >= 5 fois : 15,093

Exemple de plainte transformée :
['[DATE]', 'switch', 'phone', 'carry', '[PII]', '[PII]', '[PII]', 'pay', 'rest', 'phone', 'charge', 'leave', 'accessory', '[DATE]', 'rest', 'balance', 'owe', '[PII]', 'pay', 'check', 'clear', '[DATE]', 'month', 'start', 'get', 'call', 'debt', 'collector', 'time', 'day']

Les 20 mots les plus fréquents :
[('[PII]', 585114), ('not', 272550), ('account', 191222), ('[NUM]', 185549), ('[DATE]', 158671), ('credit', 149709), ('[MONEY]', 113362), ('payment', 112584), ('report', 104964), ('loan', 81142), ('pay', 76784), ('information', 74451), ('bank', 70538), ('receive', 70310), ('no', 69181), ('tell', 67489), ('time', 65145), ('card', 62838), ('call', 62786), ('send', 60652)]


In [3]:
from gensim.models import Word2Vec, FastText

PARAMS = dict(vector_size=100, window=5, min_count=5, sg=1, sample=1e-3,
              negative=10, epochs=10, workers=4, seed=42)

def train_w2v():
    return Word2Vec(sentences, **PARAMS).wv          # .wv = uniquement la carte des mots

def train_ft():
    return FastText(sentences, min_n=3, max_n=6, bucket=200_000, **PARAMS).wv

w2v = cached("w2v_sg_d100_w5.joblib", train_w2v)
ft  = cached("ft_sg_d100_w5_b200k.joblib", train_ft)
print(f"Word2Vec : {len(w2v.key_to_index):,} mots sur la carte, {w2v.vector_size} nombres par mot")
print(f"FastText : {len(ft.key_to_index):,} mots sur la carte (+ tous les mots inconnus via leurs morceaux)")
print("\nPosition du mot 'car' (10 premiers nombres sur 100) :")
print(np.round(w2v["car"][:10], 3))

[cache] w2v_sg_d100_w5.joblib calculé et sauvegardé (445 s)
[cache] ft_sg_d100_w5_b200k.joblib calculé et sauvegardé (720 s)
Word2Vec : 15,093 mots sur la carte, 100 nombres par mot
FastText : 15,093 mots sur la carte (+ tous les mots inconnus via leurs morceaux)

Position du mot 'car' (10 premiers nombres sur 100) :
[-0.14  -0.141  0.199  0.103 -0.052  0.049  0.142  0.178  0.128 -0.208]


In [4]:
print("=== Voisins les plus proches ===")
for w in ["car", "foreclosure", "equifax", "overdraft", "fraudulent", "not"]:
    print(f"\n{w}")
    print("  Word2Vec :", [x for x, _ in w2v.most_similar(w, topn=6)])
    print("  FastText :", [x for x, _ in ft.most_similar(w, topn=6)])

print("\n=== Fautes de frappe (mots rares ou jamais vus) ===")
for w in ["fradulent", "forclosure", "equifaxx", "repossesed"]:
    connu = w in w2v.key_to_index
    print(f"\n{w}  (connu de Word2Vec : {connu})")
    print("  FastText :", [x for x, _ in ft.most_similar(w, topn=5)])

=== Voisins les plus proches ===

car
  Word2Vec : ['vehicle', 'truck', 'dealership', 'suv', 'jeep', 'drivable']
  FastText : ['vehicle', 'vehicle-', 'truck', 'vehical', 'dealership', 'vechicle']

foreclosure
  Word2Vec : ['foreclose', 'forclosure', 'modification', 'nod', 'tro', 'adjourn']
  FastText : ['forecloser', 'foreclose', 'forclosure', 'forclosre', 'forclouser', 'forcloser']

equifax
  Word2Vec : ['experian', 'transunion', 'tranunion', 'bureaus', '/equifax', 'eqifax']
  FastText : ['equifax-', '-equifax', '/equifax', 'equifax/', 'experian', 'transunion']

overdraft
  Word2Vec : ['overdraw', 'overdrafte', 'nsf', 'overdrawn', 'od', 'nfs']
  FastText : ['overdrafts', 'overdrafted', 'overdrafte', 'overdraw', 'overdrawn', 'nsf']

fraudulent
  Word2Vec : ['unauthorized', 'fradulent', 'fradulant', 'fruadulant', 'fraudelent', 'fraudulent/']
  FastText : ['fraudulent/', 'fruadulent', 'fraudulenty', 'fradulent', 'fraudelent', 'fraudlent']

not
  Word2Vec : ['never', 'nothing', 'nor', 'no

In [5]:
from itertools import permutations

RELATIONS = {
    "entreprise -> produit": [("navient", "student"), ("ocwen", "mortgage"), ("midland", "debt"),
                              ("equifax", "report"), ("netspend", "prepaid"), ("synchrony", "card"),
                              ("santander", "vehicle"), ("paypal", "transfer"), ("chime", "bank")],
    "problème -> produit":   [("foreclosure", "mortgage"), ("repossess", "vehicle"), ("overdraft", "checking"),
                              ("forbearance", "student"), ("inquiry", "report")],
    "loi -> domaine":        [("fcra", "report"), ("fdcpa", "debt"), ("respa", "escrow"), ("tila", "disclosure")],
}

# Questions a:b :: c:? -> attendu d, seulement si les 4 mots sont sur la carte Word2Vec
VOCAB = set(w2v.key_to_index)
questions = [(cat, a, b, c, d)
             for cat, pairs in RELATIONS.items()
             for (a, b), (c, d) in permutations(pairs, 2)
             if {a, b, c, d} <= VOCAB]
absents = sorted({w for pairs in RELATIONS.values() for p in pairs for w in p} - VOCAB)
print(f"{len(questions)} analogies testées | mots absents de la carte : {absents}")

def norm(w):
    return w.strip("/-.,")          # 'report/' compte comme 'report'

def eval_analogies(kv, name):
    rows = []
    for cat, a, b, c, d in questions:
        top = [norm(w) for w, _ in kv.most_similar(positive=[b, c], negative=[a], topn=5)]
        rows.append({"modèle": name, "catégorie": cat, "question": f"{a}:{b} :: {c}:?",
                     "attendu": d, "top1": top[0], "ok@1": top[0] == d, "ok@5": d in top})
    return pl.DataFrame(rows)

ana = pl.concat([eval_analogies(w2v, "Word2Vec"), eval_analogies(ft, "FastText")])

summary = (ana.group_by("modèle", "catégorie")
              .agg(pl.len().alias("n"), pl.col("ok@1").mean().round(3), pl.col("ok@5").mean().round(3))
              .sort("catégorie", "modèle"))
display(summary)
display(ana.group_by("modèle").agg(pl.col("ok@1").mean().round(3), pl.col("ok@5").mean().round(3)))

print("\nExemples (Word2Vec) :")
with pl.Config(tbl_rows=15, fmt_str_lengths=40):
    display(ana.filter(pl.col("modèle") == "Word2Vec").sample(12, seed=1)
               .select("question", "attendu", "top1", "ok@5"))

ana.write_csv(REPORTS / "analogies_detail.csv")
summary.write_csv(REPORTS / "analogies_summary.csv")

104 analogies testées | mots absents de la carte : []


modèle,catégorie,n,ok@1,ok@5
str,str,u32,f64,f64
"""FastText""","""entreprise -> produit""",72,0.056,0.194
"""Word2Vec""","""entreprise -> produit""",72,0.097,0.292
"""FastText""","""loi -> domaine""",12,0.0,0.167
"""Word2Vec""","""loi -> domaine""",12,0.0,0.0
"""FastText""","""problème -> produit""",20,0.0,0.1
"""Word2Vec""","""problème -> produit""",20,0.0,0.15


modèle,ok@1,ok@5
str,f64,f64
"""Word2Vec""",0.067,0.231
"""FastText""",0.038,0.173



Exemples (Word2Vec) :


question,attendu,top1,ok@5
str,str,str,bool
"""forbearance:student :: foreclosure:?""","""mortgage""","""lawsuite""",false
"""repossess:vehicle :: overdraft:?""","""checking""","""overdrawn""",false
"""ocwen:mortgage :: synchrony:?""","""card""","""sychrony""",false
"""repossess:vehicle :: forbearance:?""","""student""","""forebearance""",false
"""midland:debt :: paypal:?""","""transfer""","""ebay""",false
"""paypal:transfer :: chime:?""","""bank""","""instantaneously""",false
"""tila:disclosure :: fdcpa:?""","""debt""","""representations""",false
"""paypal:transfer :: ocwen:?""","""mortgage""","""owen""",false
"""midland:debt :: ocwen:?""","""mortgage""","""sps""",false


In [6]:
ODD = [  # le dernier mot de chaque liste est l'intrus attendu
    ["equifax", "experian", "transunion", "navient"],
    ["navient", "nelnet", "mohela", "ocwen"],
    ["mortgage", "escrow", "foreclosure", "overdraft"],
    ["car", "truck", "vehicle", "house"],
    ["paypal", "venmo", "zelle", "escrow"],
    ["prepaid", "netspend", "gift", "foreclosure"],
    ["debt", "collector", "collection", "appraisal"],
    ["fraud", "unauthorized", "scam", "refinance"],
    ["overdraft", "deposit", "checking", "repossess"],
    ["forbearance", "deferment", "repayment", "dealership"],
]

rows = []
for words in ODD:
    if not set(words) <= VOCAB:
        print("ignoré (mot absent) :", words); continue
    rows.append({"mots": ", ".join(words), "intrus attendu": words[-1],
                 "Word2Vec": w2v.doesnt_match(words), "FastText": ft.doesnt_match(words)})
odd = pl.DataFrame(rows)
with pl.Config(fmt_str_lengths=60, tbl_width_chars=200):
    display(odd)
print(f"Word2Vec : {(odd['Word2Vec'] == odd['intrus attendu']).sum()}/{odd.height}")
print(f"FastText : {(odd['FastText'] == odd['intrus attendu']).sum()}/{odd.height}")
odd.write_csv(REPORTS / "odd_one_out.csv")

mots,intrus attendu,Word2Vec,FastText
str,str,str,str
"""equifax, experian, transunion, navient""","""navient""","""navient""","""navient"""
"""navient, nelnet, mohela, ocwen""","""ocwen""","""ocwen""","""ocwen"""
"""mortgage, escrow, foreclosure, overdraft""","""overdraft""","""overdraft""","""overdraft"""
"""car, truck, vehicle, house""","""house""","""house""","""house"""
"""paypal, venmo, zelle, escrow""","""escrow""","""escrow""","""escrow"""
"""prepaid, netspend, gift, foreclosure""","""foreclosure""","""foreclosure""","""foreclosure"""
"""debt, collector, collection, appraisal""","""appraisal""","""appraisal""","""appraisal"""
"""fraud, unauthorized, scam, refinance""","""refinance""","""refinance""","""refinance"""
"""overdraft, deposit, checking, repossess""","""repossess""","""repossess""","""repossess"""


Word2Vec : 10/10
FastText : 10/10


In [7]:
def eval_analogies_m(kv, name, method):
    f = kv.most_similar if method == "3CosAdd" else kv.most_similar_cosmul
    rows = []
    for cat, a, b, c, d in questions:
        top = [norm(w) for w, _ in f(positive=[b, c], negative=[a], topn=5)]
        rows.append({"modèle": name, "formule": method, "catégorie": cat,
                     "question": f"{a}:{b} :: {c}:?", "attendu": d, "top1": top[0],
                     "ok@1": top[0] == d, "ok@5": d in top})
    return pl.DataFrame(rows)

ana2 = pl.concat([eval_analogies_m(kv, n, m)
                  for kv, n in [(w2v, "Word2Vec"), (ft, "FastText")]
                  for m in ["3CosAdd", "3CosMul"]])

display(ana2.group_by("modèle", "formule")
            .agg(pl.col("ok@1").mean().round(3), pl.col("ok@5").mean().round(3))
            .sort("modèle", "formule"))
display(ana2.group_by("modèle", "formule", "catégorie")
            .agg(pl.col("ok@5").mean().round(3))
            .sort("catégorie", "modèle", "formule"))

ana2.write_csv(REPORTS / "analogies_detail.csv")

modèle,formule,ok@1,ok@5
str,str,f64,f64
"""FastText""","""3CosAdd""",0.038,0.173
"""FastText""","""3CosMul""",0.058,0.144
"""Word2Vec""","""3CosAdd""",0.067,0.231
"""Word2Vec""","""3CosMul""",0.038,0.173


modèle,formule,catégorie,ok@5
str,str,str,f64
"""FastText""","""3CosAdd""","""entreprise -> produit""",0.194
"""FastText""","""3CosMul""","""entreprise -> produit""",0.153
"""Word2Vec""","""3CosAdd""","""entreprise -> produit""",0.292
"""Word2Vec""","""3CosMul""","""entreprise -> produit""",0.208
"""FastText""","""3CosAdd""","""loi -> domaine""",0.167
…,…,…,…
"""Word2Vec""","""3CosMul""","""loi -> domaine""",0.0
"""FastText""","""3CosAdd""","""problème -> produit""",0.1
"""FastText""","""3CosMul""","""problème -> produit""",0.1


In [8]:
val_sentences = [t.split() for t in val["text_tfidf"].to_list()]

# IDF appris sur le train uniquement : log(nombre de plaintes / nombre de plaintes contenant le mot)
N_DOCS = len(sentences)
doc_freq = {}
for s in sentences:
    for w in set(s):
        doc_freq[w] = doc_freq.get(w, 0) + 1
IDF = {w: np.log(N_DOCS / c) for w, c in doc_freq.items()}
IDF_MAX = np.log(N_DOCS)                       # pour un mot jamais vu en train

def doc_vectors(kv, docs, weighting="mean", use_oov=False):
    """Une plainte = moyenne (simple ou pondérée IDF) des positions de ses mots."""
    index, V = kv.key_to_index, kv.vectors
    oov_cache = {}
    out = np.zeros((len(docs), kv.vector_size), dtype=np.float32)
    n_empty = 0
    for i, words in enumerate(docs):
        vecs, weights = [], []
        for w in words:
            if w in index:
                v = V[index[w]]
            elif use_oov:                       # FastText : position calculée à partir des morceaux du mot
                if w not in oov_cache:
                    oov_cache[w] = kv[w]
                v = oov_cache[w]
            else:
                continue
            vecs.append(v)
            weights.append(IDF.get(w, IDF_MAX) if weighting == "idf" else 1.0)
        if vecs:
            out[i] = np.average(vecs, axis=0, weights=weights)
        else:
            n_empty += 1
    print(f"  {len(docs):,} plaintes, dont {n_empty} sans aucun mot connu")
    return out

CONFIGS = {
    "Word2Vec, moyenne":                (w2v, "mean", False, "docvec_w2v_mean.joblib"),
    "Word2Vec, moyenne pondérée IDF":   (w2v, "idf",  False, "docvec_w2v_idf.joblib"),
    "FastText, moyenne (avec inconnus)": (ft,  "mean", True,  "docvec_ft_mean_oov.joblib"),
}
feats = {}
for name, (kv, wt, oov, fname) in CONFIGS.items():
    print(name)
    feats[name] = cached(fname, lambda kv=kv, wt=wt, oov=oov:
                         (doc_vectors(kv, sentences, wt, oov), doc_vectors(kv, val_sentences, wt, oov)))
print("\nForme pour le train :", feats["Word2Vec, moyenne"][0].shape)

Word2Vec, moyenne
  100,000 plaintes, dont 1 sans aucun mot connu
  50,000 plaintes, dont 0 sans aucun mot connu
[cache] docvec_w2v_mean.joblib calculé et sauvegardé (21 s)
Word2Vec, moyenne pondérée IDF
  100,000 plaintes, dont 1 sans aucun mot connu
  50,000 plaintes, dont 0 sans aucun mot connu
[cache] docvec_w2v_idf.joblib calculé et sauvegardé (23 s)
FastText, moyenne (avec inconnus)
  100,000 plaintes, dont 0 sans aucun mot connu
  50,000 plaintes, dont 0 sans aucun mot connu
[cache] docvec_ft_mean_oov.joblib calculé et sauvegardé (22 s)

Forme pour le train : (100000, 100)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score

y_train = train["product"].to_numpy()
y_val   = val["product"].to_numpy()
CLASSES = sorted(set(y_train))

# Mêmes poids de classe qu'en L2 (alpha = 0.75)
POOL = {"credit_reporting": 569_176, "debt_collection": 217_966, "credit_card": 121_859,
        "mortgage": 112_942, "checking_savings": 101_743, "student_loan": 39_948,
        "money_transfer": 38_190, "vehicle_loan": 33_968, "payday_personal_loan": 23_654,
        "prepaid_card": 12_417}
N_TRAIN = dict(train.group_by("product").len().iter_rows())
def class_weights(alpha_target):
    raw = {c: POOL[c] ** (alpha_target - 0.5) for c in CLASSES}
    mean = sum(raw[c] * N_TRAIN[c] for c in CLASSES) / sum(N_TRAIN.values())
    return {c: raw[c] / mean for c in CLASSES}
CW = class_weights(0.75)

def run_extrinsic():
    out = {}
    for name, (Xtr, Xva) in feats.items():
        sc = StandardScaler().fit(Xtr)
        Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)
        best = None
        for C in [0.1, 1.0, 10.0]:                     # réglage de C sur la val, comme en L2
            m = LogisticRegression(C=C, max_iter=3000, class_weight=CW).fit(Xtr_s, y_train)
            p = m.predict(Xva_s)
            f = f1_score(y_val, p, average="macro")
            if best is None or f > best["macro_f1"]:
                best = {"C": C, "macro_f1": f, "accuracy": accuracy_score(y_val, p), "pred": p}
        out[name] = best
        print(f"{name} : macro-F1 {best['macro_f1']:.3f} (C = {best['C']})")
    return out

ext = cached("l3_extrinsic.joblib", run_extrinsic)

table = pl.DataFrame(
    [{"représentation": n, "dimensions": 100, "accuracy": round(r["accuracy"], 3),
      "macro_f1": round(r["macro_f1"], 3)} for n, r in ext.items()]
    + [{"représentation": "TF-IDF + régression logistique (L2)", "dimensions": 200_000, "accuracy": 0.848, "macro_f1": 0.752},
       {"représentation": "TF-IDF + SVM réglé (L2)",             "dimensions": 200_000, "accuracy": 0.856, "macro_f1": 0.763}]
).sort("macro_f1", descending=True)
display(table)
table.write_csv(REPORTS / "extrinsic_comparison.csv")

# F1 par classe : meilleure version embeddings vs SVM L2
L2_F1 = {"checking_savings": 0.812, "credit_card": 0.745, "credit_reporting": 0.922, "debt_collection": 0.715,
         "money_transfer": 0.706, "mortgage": 0.873, "payday_personal_loan": 0.542, "prepaid_card": 0.735,
         "student_loan": 0.868, "vehicle_loan": 0.708}
best_name = max(ext, key=lambda n: ext[n]["macro_f1"])
f1_emb = f1_score(y_val, ext[best_name]["pred"], labels=CLASSES, average=None)
per_class = pl.DataFrame({"classe": CLASSES,
                          "SVM TF-IDF (L2)": [L2_F1[c] for c in CLASSES],
                          best_name: np.round(f1_emb, 3)}) \
              .with_columns((pl.col(best_name) - pl.col("SVM TF-IDF (L2)")).round(3).alias("écart")) \
              .sort("écart")
display(per_class)
per_class.write_csv(REPORTS / "extrinsic_per_class.csv")

Word2Vec, moyenne : macro-F1 0.701 (C = 0.1)
Word2Vec, moyenne pondérée IDF : macro-F1 0.678 (C = 0.1)
FastText, moyenne (avec inconnus) : macro-F1 0.699 (C = 0.1)
[cache] l3_extrinsic.joblib calculé et sauvegardé (121 s)


représentation,dimensions,accuracy,macro_f1
str,i64,f64,f64
"""TF-IDF + SVM réglé (L2)""",200000,0.856,0.763
"""TF-IDF + régression logistique…",200000,0.848,0.752
"""Word2Vec, moyenne""",100,0.829,0.701
"""FastText, moyenne (avec inconn…",100,0.829,0.699
"""Word2Vec, moyenne pondérée IDF""",100,0.818,0.678


classe,SVM TF-IDF (L2),"Word2Vec, moyenne",écart
str,f64,f64,f64
"""payday_personal_loan""",0.542,0.402,-0.14
"""vehicle_loan""",0.708,0.613,-0.095
"""prepaid_card""",0.735,0.652,-0.083
"""credit_card""",0.745,0.69,-0.055
"""money_transfer""",0.706,0.651,-0.055
"""student_loan""",0.868,0.817,-0.051
"""mortgage""",0.873,0.823,-0.05
"""checking_savings""",0.812,0.769,-0.043
"""debt_collection""",0.715,0.678,-0.037


In [10]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")   # 400k mots, 100 nombres par mot, appris sur Wikipédia + presse
print(f"GloVe : {len(glove.key_to_index):,} mots")

# 1. Couverture : part de nos mots (et de nos occurrences) que GloVe connaît
our_vocab = list(w2v.key_to_index)
known = [w for w in our_vocab if w in glove.key_to_index]
occ_total = sum(vocab[w] for w in our_vocab)
occ_known = sum(vocab[w] for w in known)
print(f"Mots de notre carte connus de GloVe : {len(known):,}/{len(our_vocab):,} ({len(known)/len(our_vocab):.1%})")
print(f"Part des occurrences couvertes      : {occ_known/occ_total:.1%}")
missing = sorted((w for w in our_vocab if w not in glove.key_to_index), key=lambda w: -vocab[w])[:30]
print("Mots fréquents inconnus de GloVe :", missing)

# 2. Voisins des mots dont le sens change entre la finance et l'anglais général
print("\n=== Même mot, deux cartes ===")
for w in ["charge", "collection", "statement", "dispute", "balance", "default", "servicer", "forbearance", "escrow", "chime"]:
    print(f"\n{w}")
    print("  Notre Word2Vec :", [x for x, _ in w2v.most_similar(w, topn=6)] if w in w2v.key_to_index else "absent")
    print("  GloVe          :", [x for x, _ in glove.most_similar(w, topn=6)] if w in glove.key_to_index else "absent")

[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe : 400,000 mots
Mots de notre carte connus de GloVe : 12,366/15,093 (81.9%)
Part des occurrences couvertes      : 88.1%
Mots fréquents inconnus de GloVe : ['[PII]', '[NUM]', '[DATE]', '[MONEY]', 'usc_1681', 'navient', 'coinbase', 'usc_1692', 'furnisher', 'pende', 'fdcpa', 'covid', 'nationstar', 'mohela', 'scamme', 'fedloan', 'pslf', '605b', 'truist', 'covid-', 'comenity', 'venmo', 'shellpoint', 'netspend', '[URL]', 'autopay', '-day', 'loancare', 'antedate', '.i']

=== Même mot, deux cartes ===

charge
  Notre Word2Vec : ['carrying', 'charges/', 'fee', 'assess', 'overlimit', 'fess']
  GloVe          : ['charges', 'charged', 'criminal', 'for', 'charging', 'guilty']

collection
  Notre Word2Vec : ['collecton', 'waypoint', 'debt', 'aargon', 'simon', 'medicredit']
  GloVe          : ['collections', 'works', 'catalogue', 'catalog', 'artwork', 'book']

statement
  Notre Word2Vec : ['statment', 'ebill', '

In [11]:
# 3a. Analogies : seulement les questions dont les 4 mots sont connus des DEUX cartes (comparaison juste)
GV = set(glove.key_to_index)
q_common = [q for q in questions if set(q[1:]) <= GV]
print(f"Analogies communes : {len(q_common)}/{len(questions)}")

def eval_on(kv, name, qs):
    rows = []
    for cat, a, b, c, d in qs:
        top = [norm(w) for w, _ in kv.most_similar(positive=[b, c], negative=[a], topn=5)]
        rows.append({"modèle": name, "catégorie": cat, "question": f"{a}:{b} :: {c}:?",
                     "attendu": d, "top1": top[0], "ok@1": top[0] == d, "ok@5": d in top})
    return pl.DataFrame(rows)

ana_g = pl.concat([eval_on(w2v, "Word2Vec (plaintes)", q_common), eval_on(glove, "GloVe (Wikipédia)", q_common)])
display(ana_g.group_by("modèle").agg(pl.len().alias("n"), pl.col("ok@1").mean().round(3), pl.col("ok@5").mean().round(3)))
display(ana_g.group_by("modèle", "catégorie").agg(pl.len().alias("n"), pl.col("ok@5").mean().round(3)).sort("catégorie", "modèle"))

# 3b. Intrus
rows = []
for words in ODD:
    if set(words) <= GV:
        rows.append({"mots": ", ".join(words), "attendu": words[-1],
                     "Word2Vec": w2v.doesnt_match(words), "GloVe": glove.doesnt_match(words)})
    else:
        print("GloVe ne connaît pas :", [w for w in words if w not in GV])
odd_g = pl.DataFrame(rows)
with pl.Config(fmt_str_lengths=60, tbl_width_chars=200):
    display(odd_g)
print(f"Word2Vec : {(odd_g['Word2Vec'] == odd_g['attendu']).sum()}/{odd_g.height}   "
      f"GloVe : {(odd_g['GloVe'] == odd_g['attendu']).sum()}/{odd_g.height}")

# 4. Classification : moyenne GloVe + même régression logistique
def run_glove_extrinsic():
    Xtr = doc_vectors(glove, sentences, "mean", False)
    Xva = doc_vectors(glove, val_sentences, "mean", False)
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)
    best = None
    for C in [0.1, 1.0, 10.0]:
        m = LogisticRegression(C=C, max_iter=3000, class_weight=CW).fit(Xtr_s, y_train)
        p = m.predict(Xva_s)
        f = f1_score(y_val, p, average="macro")
        if best is None or f > best["macro_f1"]:
            best = {"C": C, "macro_f1": f, "accuracy": accuracy_score(y_val, p), "pred": p}
    return best

ext_g = cached("l3_glove_extrinsic.joblib", run_glove_extrinsic)
print(f"\nGloVe, moyenne : macro-F1 {ext_g['macro_f1']:.3f} (accuracy {ext_g['accuracy']:.3f}, C = {ext_g['C']})")
print(f"Word2Vec, moyenne (notre carte) : macro-F1 {ext['Word2Vec, moyenne']['macro_f1']:.3f}")

ana_g.write_csv(REPORTS / "analogies_glove_vs_w2v.csv")
odd_g.write_csv(REPORTS / "odd_one_out_glove_vs_w2v.csv")

Analogies communes : 64/104


modèle,n,ok@1,ok@5
str,u32,f64,f64
"""Word2Vec (plaintes)""",64,0.016,0.125
"""GloVe (Wikipédia)""",64,0.0,0.016


modèle,catégorie,n,ok@5
str,str,u32,f64
"""GloVe (Wikipédia)""","""entreprise -> produit""",42,0.0
"""Word2Vec (plaintes)""","""entreprise -> produit""",42,0.119
"""GloVe (Wikipédia)""","""loi -> domaine""",2,0.0
"""Word2Vec (plaintes)""","""loi -> domaine""",2,0.0
"""GloVe (Wikipédia)""","""problème -> produit""",20,0.05
"""Word2Vec (plaintes)""","""problème -> produit""",20,0.15


GloVe ne connaît pas : ['navient']
GloVe ne connaît pas : ['navient', 'mohela']
GloVe ne connaît pas : ['venmo']
GloVe ne connaît pas : ['netspend']


mots,attendu,Word2Vec,GloVe
str,str,str,str
"""mortgage, escrow, foreclosure, overdraft""","""overdraft""","""overdraft""","""overdraft"""
"""car, truck, vehicle, house""","""house""","""house""","""house"""
"""debt, collector, collection, appraisal""","""appraisal""","""appraisal""","""appraisal"""
"""fraud, unauthorized, scam, refinance""","""refinance""","""refinance""","""refinance"""
"""overdraft, deposit, checking, repossess""","""repossess""","""repossess""","""repossess"""
"""forbearance, deferment, repayment, dealership""","""dealership""","""dealership""","""dealership"""


Word2Vec : 6/6   GloVe : 6/6
  100,000 plaintes, dont 9 sans aucun mot connu
  50,000 plaintes, dont 2 sans aucun mot connu
[cache] l3_glove_extrinsic.joblib calculé et sauvegardé (61 s)

GloVe, moyenne : macro-F1 0.586 (accuracy 0.774, C = 10.0)
Word2Vec, moyenne (notre carte) : macro-F1 0.701


In [12]:
import re, math, random
from collections import Counter

full_tr = pl.read_parquet(f"{FEAT}/train_text.parquet", columns=["product", "text_full"])
full_va = pl.read_parquet(f"{FEAT}/val_text.parquet",   columns=["product", "text_full"])
PRODUCTS = sorted(full_tr["product"].unique().to_list())

TOKEN_RE = re.compile(r"\[[A-Z]+\]|[a-z0-9_]+(?:'[a-z]+)?|[.!?]")   # tokens spéciaux, mots (avec n't), fin de phrase
def tokenize(t):
    return TOKEN_RE.findall(t)

N_PER_CLASS_TRAIN, N_PER_CLASS_VAL = 3_600, 200      # même quantité de texte pour chaque produit
lm_train = {p: [tokenize(t) for t in full_tr.filter(pl.col("product") == p)
                                            .sample(N_PER_CLASS_TRAIN, seed=42)["text_full"]]
            for p in PRODUCTS}
lm_val = {p: [tokenize(t) for t in full_va.filter(pl.col("product") == p)
                                          .sample(N_PER_CLASS_VAL, seed=42)["text_full"]]
          for p in PRODUCTS}

# Vocabulaire commun à tous les modèles : mots vus au moins 3 fois ; les autres deviennent <unk>
counts = Counter(w for docs in lm_train.values() for d in docs for w in d)
LM_VOCAB = {w for w, c in counts.items() if c >= 3} | {"</s>", "<unk>"}
def to_vocab(doc):
    return [w if w in LM_VOCAB else "<unk>" for w in doc]
lm_train = {p: [to_vocab(d) for d in docs] for p, docs in lm_train.items()}
lm_val   = {p: [to_vocab(d) for d in docs] for p, docs in lm_val.items()}

n_tok = sum(len(d) for docs in lm_train.values() for d in docs)
print(f"Vocabulaire : {len(LM_VOCAB):,} mots | {n_tok:,} mots d'apprentissage au total")
print("Exemple :", " ".join(lm_train["mortgage"][0][:40]))

Vocabulaire : 16,310 mots | 8,148,871 mots d'apprentissage au total
Exemple : dear cfpb team i have a mortgage with citadel servicing corporation . due to the coronavirus lockdown we were forced to apply for forbearance which citadel approved from [DATE] until [DATE] . we then agreed on a payback plan which


In [13]:
class KneserNeyTrigram:
    """Trigramme à lissage Kneser-Ney interpolé, remise d = 0.75."""
    def __init__(self, vocab_size, d=0.75):
        self.V, self.d = vocab_size, d

    def fit(self, docs):
        self.tri = Counter()
        for toks in docs:
            s = ["<s>", "<s>"] + toks + ["</s>"]
            for i in range(2, len(s)):
                self.tri[(s[i-2], s[i-1], s[i])] += 1
        # Niveau 3 : comptes des contextes (u, v)
        self.ctx_count, self.ctx_types, self.cont_bi = Counter(), Counter(), Counter()
        for (u, v, w), c in self.tri.items():
            self.ctx_count[(u, v)] += c          # combien de fois le contexte (u, v) a été vu
            self.ctx_types[(u, v)] += 1          # combien de mots différents l'ont suivi
            self.cont_bi[(v, w)] += 1            # après combien de mots u différents la paire (v, w) apparaît
        # Niveau 2 : comptes de continuation
        self.cont_ctx, self.cont_types, cont_uni = Counter(), Counter(), Counter()
        for (v, w), c in self.cont_bi.items():
            self.cont_ctx[v] += c
            self.cont_types[v] += 1
            cont_uni[w] += 1                     # après combien de mots différents w apparaît
        total = sum(cont_uni.values())
        self.p1 = {w: c / total for w, c in cont_uni.items()}
        return self

    def prob(self, u, v, w):
        d = self.d
        p = 0.99 * self.p1.get(w, 0.0) + 0.01 / self.V                        # niveau 1 : continuation (+ un peu d'uniforme)
        cc = self.cont_ctx.get(v, 0)
        if cc:                                                                # niveau 2 : contexte (v)
            p = (max(self.cont_bi.get((v, w), 0) - d, 0) + d * self.cont_types[v] * p) / cc
        c = self.ctx_count.get((u, v), 0)
        if c:                                                                 # niveau 3 : contexte (u, v)
            p = (max(self.tri.get((u, v, w), 0) - d, 0) + d * self.ctx_types[(u, v)] * p) / c
        return p

    def doc_logprob(self, toks):
        s = ["<s>", "<s>"] + toks + ["</s>"]
        return sum(math.log(self.prob(s[i-2], s[i-1], s[i])) for i in range(2, len(s))), len(s) - 2

t0 = time.time()
LMS = {p: KneserNeyTrigram(len(LM_VOCAB)).fit(docs) for p, docs in lm_train.items()}
print(f"10 modèles appris en {time.time() - t0:.0f} s")

# Vérification : les probabilités d'un contexte doivent sommer à 1
m = LMS["mortgage"]
s = sum(m.prob("late", "payment", w) for w in LM_VOCAB)
print(f"Somme des probabilités après 'late payment' : {s:.4f}  (doit être proche de 1)")
print("Top 5 après 'late payment' (mortgage) :",
      sorted(((w, round(m.prob('late', 'payment', w), 3)) for w in LM_VOCAB), key=lambda x: -x[1])[:5])

10 modèles appris en 15 s
Somme des probabilités après 'late payment' : 1.0000  (doit être proche de 1)
Top 5 après 'late payment' (mortgage) : [('.', 0.15), ('on', 0.113), ('and', 0.088), ('in', 0.074), ('for', 0.049)]


In [14]:
def run_perplexity():
    # logp[modèle][produit du texte] = liste (log-probabilité, nb de mots) pour chaque plainte
    logp = {m: {p: [LMS[m].doc_logprob(d) for d in lm_val[p]] for p in PRODUCTS} for m in PRODUCTS}
    return logp

logp = cached("l3_ngram_logp.joblib", run_perplexity)

# Tableau de perplexité : ligne = modèle, colonne = produit des plaintes lues
ppl = {m: {p: math.exp(-sum(lp for lp, _ in logp[m][p]) / sum(n for _, n in logp[m][p])) for p in PRODUCTS}
       for m in PRODUCTS}
ppl_df = pl.DataFrame([{"modèle": m, **{p: round(ppl[m][p], 1) for p in PRODUCTS}} for m in PRODUCTS])
with pl.Config(tbl_cols=12, tbl_width_chars=250):
    display(ppl_df)
ppl_df.write_csv(REPORTS / "ngram_perplexity_matrix.csv")

# Classification : pour chaque plainte, le produit dont le modèle est le moins surpris
y_true, y_pred = [], []
for p in PRODUCTS:
    for i in range(len(lm_val[p])):
        scores = {m: logp[m][p][i][0] for m in PRODUCTS}
        y_true.append(p); y_pred.append(max(scores, key=scores.get))
print(f"\nClassification par perplexité (échantillon équilibré, 200 plaintes par produit) :")
print(f"  accuracy {accuracy_score(y_true, y_pred):.3f} | macro-F1 {f1_score(y_true, y_pred, average='macro'):.3f}")
print(classification_report(y_true, y_pred, digits=3) if 'classification_report' in dir() else "")

[cache] l3_ngram_logp.joblib calculé et sauvegardé (12 s)


modèle,checking_savings,credit_card,credit_reporting,debt_collection,money_transfer,mortgage,payday_personal_loan,prepaid_card,student_loan,vehicle_loan
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""checking_savings""",98.0,112.9,257.7,205.9,116.5,189.6,149.9,99.2,196.4,159.5
"""credit_card""",121.2,91.2,134.0,152.4,140.6,186.0,132.9,99.1,173.7,135.1
"""credit_reporting""",203.4,157.5,29.8,104.5,231.5,226.1,176.1,190.2,193.0,160.6
"""debt_collection""",179.9,144.7,78.1,83.9,198.9,191.2,152.0,161.0,183.4,137.7
"""money_transfer""",108.9,126.6,298.4,214.0,94.4,202.6,161.3,102.0,216.6,171.1
"""mortgage""",150.8,140.1,234.7,169.7,171.1,106.5,135.3,160.1,150.2,135.1
"""payday_personal_loan""",141.7,115.6,123.8,141.7,156.7,158.6,109.0,127.3,156.6,110.9
"""prepaid_card""",124.9,127.0,280.2,255.0,133.0,238.1,178.4,73.2,229.8,187.3
"""student_loan""",181.1,151.0,197.8,173.1,206.6,180.0,145.2,170.5,91.6,157.8



Classification par perplexité (échantillon équilibré, 200 plaintes par produit) :
  accuracy 0.707 | macro-F1 0.704



In [15]:
GEN_VOCAB = [w for w in LM_VOCAB if w != "<unk>"]

def generate(model, prompt, n=35, seed=0):
    rng = random.Random(seed)
    s = ["<s>", "<s>"] + prompt.split()
    for _ in range(n):
        weights = [model.prob(s[-2], s[-1], w) for w in GEN_VOCAB]
        w = rng.choices(GEN_VOCAB, weights=weights)[0]
        if w == "</s>":
            break
        s.append(w)
    return " ".join(s[2:])

for p in ["mortgage", "debt_collection", "student_loan", "prepaid_card"]:
    print(f"\n[{p}]")
    for seed in [1, 2]:
        print("  -", generate(LMS[p], "i called the", seed=seed))


[mortgage]
  - i called the condition it was a refinance and stuck in that time i had the through and damage to release it will they be paid for a hardship funding for [PII] . so as to how a
  - i called the mortgage insurance cancelled off or in the process but instead of a very malpractices the does not include any debt truly appreciate lies . if i had been so slow lot to two businesses when

[debt_collection]
  - i called the company [NUM] loans liens against ask them if this month relationship refusal identity thief where notation on social security agreement between myself and my account . this account and had no longer were never disclosed
  - i called the trustees who they are papers with acceptable user policy . i did not respond . none of the [PII] [PII] mi 1681i a you are not subparagraph a by took my taxes how does not

[student_loan]
  - i called the company making payments on my student loans with fedloan take in [PII] of florida started with a copy of this change and not

In [16]:
# Voisins des mots ambigus : notre carte contre GloVe
rows = []
for w in ["charge", "collection", "statement", "dispute", "balance", "default", "servicer", "forbearance", "escrow", "chime"]:
    rows.append({"mot": w,
                 "word2vec_plaintes": ", ".join(x for x, _ in w2v.most_similar(w, topn=6)),
                 "glove_wikipedia": ", ".join(x for x, _ in glove.most_similar(w, topn=6)) if w in glove.key_to_index else "absent"})
pl.DataFrame(rows).write_csv(REPORTS / "domain_shift_neighbors.csv")

# Textes générés par les modèles n-grammes
gens = [{"produit": p, "seed": s, "texte": generate(LMS[p], "i called the", seed=s)}
        for p in ["mortgage", "debt_collection", "student_loan", "prepaid_card"] for s in [1, 2]]
pl.DataFrame(gens).write_csv(REPORTS / "ngram_generations.csv")

# Embeddings au format standard, pour les réutiliser en L4
MODELS = Path("/kaggle/working/models"); MODELS.mkdir(exist_ok=True)
w2v.save_word2vec_format(str(MODELS / "w2v_complaints_d100.txt"))
ft.save(str(MODELS / "ft_complaints_d100.kv"))
!ls -la /kaggle/working/reports_l3 /kaggle/working/models

/kaggle/working/models:
total 101612
drwxr-xr-x 2 root root     4096 Nov 13 14:48 .
drwxr-xr-x 5 root root     4096 Nov 13 14:48 ..
-rw-r--r-- 1 root root  6550732 Nov 13 14:48 ft_complaints_d100.kv
-rw-r--r-- 1 root root 80000128 Nov 13 14:48 ft_complaints_d100.kv.vectors_ngrams.npy
-rw-r--r-- 1 root root 17482530 Nov 13 14:48 w2v_complaints_d100.txt

/kaggle/working/reports_l3:
total 96
drwxr-xr-x 2 root root  4096 Nov 13 14:48 .
drwxr-xr-x 5 root root  4096 Nov 13 14:48 ..
-rw-r--r-- 1 root root 39125 Nov 13 14:41 analogies_detail.csv
-rw-r--r-- 1 root root 12565 Nov 13 14:47 analogies_glove_vs_w2v.csv
-rw-r--r-- 1 root root   278 Nov 13 14:41 analogies_summary.csv
-rw-r--r-- 1 root root  1328 Nov 13 14:48 domain_shift_neighbors.csv
-rw-r--r-- 1 root root   285 Nov 13 14:44 extrinsic_comparison.csv
-rw-r--r-- 1 root root   383 Nov 13 14:44 extrinsic_per_class.csv
-rw-r--r-- 1 root root  1728 Nov 13 14:48 ngram_generations.csv
-rw-r--r-- 1 root root   888 Nov 13 14:47 ngram_perplexit